In [ ]:
import pandas as pd
import numpy as np

# Upload file in Colab: click the folder icon (left) -> upload CSV
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

print("\nTarget distribution (counts):")
print(df["Churn"].value_counts())

print("\nTarget distribution (percent):")
print(df["Churn"].value_counts(normalize=True).round(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Check missing values
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(10))

# Drop rows where TotalCharges is missing
df = df.dropna(subset=["TotalCharges"]).copy()

print("\nShape after dropping missing TotalCharges:", df.shape)

TotalCharges      11
gender             0
SeniorCitizen      0
Partner            0
customerID         0
Dependents         0
tenure             0
MultipleLines      0
PhoneService       0
OnlineSecurity     0
dtype: int64

Shape after dropping missing TotalCharges: (7032, 21)


In [ ]:
df = df.drop("customerID", axis=1)

In [ ]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [ ]:
df["Churn"].value_counts()

,count
Churn,
0,5163
1,1869


In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4922, 30)
Test shape: (2110, 30)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Initialize model
log_model = LogisticRegression(max_iter=1000)

# Fit model
log_model.fit(X_train, y_train)

# Predictions
y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

# Evaluation
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))

Confusion Matrix:
 [[1382  167]
 [ 245  316]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1549
           1       0.65      0.56      0.61       561

    accuracy                           0.80      2110
   macro avg       0.75      0.73      0.74      2110
weighted avg       0.80      0.80      0.80      2110

ROC-AUC: 0.8380951887768431


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": log_model.coef_[0]
})

coefficients = coefficients.sort_values(by="Coefficient", ascending=False)

print("Top Positive Drivers (Increase Churn):")
print(coefficients.head(10))

print("\nTop Negative Drivers (Reduce Churn):")
print(coefficients.tail(10))

Top Positive Drivers (Increase Churn):
                           Feature  Coefficient
10     InternetService_Fiber optic     0.647128
28  PaymentMethod_Electronic check     0.358806
26            PaperlessBilling_Yes     0.331795
9                MultipleLines_Yes     0.304998
8   MultipleLines_No phone service     0.235376
21                 StreamingTV_Yes     0.180840
0                    SeniorCitizen     0.179021
23             StreamingMovies_Yes     0.169089
29      PaymentMethod_Mailed check     0.096496
5                      Partner_Yes     0.026694

Top Negative Drivers (Reduce Churn):
                                 Feature  Coefficient
18       TechSupport_No internet service    -0.121143
16  DeviceProtection_No internet service    -0.121143
20       StreamingTV_No internet service    -0.121143
15                      OnlineBackup_Yes    -0.216346
6                         Dependents_Yes    -0.265162
7                       PhoneService_Yes    -0.424168
19               

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Fit model
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluation
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

Confusion Matrix:
 [[1386  163]
 [ 295  266]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.89      0.86      1549
           1       0.62      0.47      0.54       561

    accuracy                           0.78      2110
   macro avg       0.72      0.68      0.70      2110
weighted avg       0.77      0.78      0.77      2110

ROC-AUC: 0.816341748859882


In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(10)

,Feature,Importance
3,TotalCharges,0.190371
1,tenure,0.171410
2,MonthlyCharges,0.164100
10,InternetService_Fiber optic,0.038658
28,PaymentMethod_Electronic check,0.036020
25,Contract_Two year,0.032695
4,gender_Male,0.028828
13,OnlineSecurity_Yes,0.027582
26,PaperlessBilling_Yes,0.025493
5,Partner_Yes,0.023594
